# Statistical Analysis of the Model Comparison

Reads the judge scores and inference metrics from Notebooks 04 and 05 and answers two questions.<br>
**Question 1:** How similar is the Student to the Teacher, per criterion?<br>
**Question 2:** Is the Student closer to the Teacher than the untrained Base model?

**Pipeline position:** 04 Inference → 05 Judge Scoring → `[06 Analysis]`<br>
**No GPU and no API key required.** Reads only from `results/`.

The Teacher is the reference throughout.<br>
Absolute compliance rates are not interpretable, because the ideal rate for `structured_steps` is unknown.<br>
Not every customer query warrants a numbered list, so a model at 100% would be over-applying the format.<br>
Agreement with the Teacher has a known maximum of 100% and is comparable across all four criteria.

Three queries are excluded: q068, q070 and q113.<br>
The Teacher response degenerated into a runaway enumeration there, which makes it unusable as a reference.<br>
Since every agreement rate is measured against the Teacher, all three models are dropped for these queries.

## 1. Setup

Install dependencies and import standard libraries.

In [1]:
import json
import os

import numpy as np
import pandas as pd
from scipy import stats

RESULTS_DIR = "../results"

MODELS          = ["base", "teacher", "student"]
REFERENCE_MODEL = "teacher"                       # every agreement rate is measured against this model
COMPARED_MODELS = ["base", "student"]

CRITERIA = ["acknowledgement", "structured_steps", "closing", "tone"]

# Dropped for all three models: the Teacher answer is unusable for these queries,
EXCLUDED_QUERIES = ["q068", "q070", "q113"]

CONFIDENCE   = 0.95
N_BOOTSTRAP  = 10_000
RANDOM_SEED  = 42        # fixed so the bootstrap intervals are reproducible across runs

print(f"numpy:  {np.__version__}")
print(f"pandas: {pd.__version__}")
print(f"scipy:  {stats.__name__.split('.')[0]} {__import__('scipy').__version__}")
print(f"Results dir: {os.path.abspath(RESULTS_DIR)}")

numpy:  2.4.4
pandas: 3.0.2
scipy:  scipy 1.18.1
Results dir: C:\Users\Battlestation\PycharmProjects\00_python-ki-advanced\distil-support-llm\results


## 1. Load Results

All sections read from a single long-format DataFrame.<br>
One row per (query, model) pair, carrying query metadata, response metrics and the four judge criteria.

Loading fails loudly if any response is missing its judge score.<br>
That guards against a stale `judge_scores.json`.<br>
Its resume logic would otherwise leave old scores attached to responses that no longer exist.

In [2]:
def load_results(results_dir, models, excluded_queries):
    """Build the single long-format DataFrame that all analyses in this notebook read from.

    Response metrics, judge scores and the per-model VRAM figure are merged into one
    shape so that no section needs a data structure of its own.

    Args:
        results_dir: Directory holding the JSON result files.
        models: Model labels to load, one `raw_generations_<label>.json` per label.
        excluded_queries: query_ids to drop across all models.

    Returns:
        DataFrame with one row per (query_id, model_label), carrying query metadata,
        response metrics, the four binary criteria, judge_total and vram_peak_gb.

    Raises:
        ValueError: If any response has no matching judge score.
    """
    def read(filename):
        with open(os.path.join(results_dir, filename), encoding="utf-8") as f:
            return json.load(f)

    generations = pd.DataFrame([
        {"model_label": label, **entry}
        for label in models
        for entry in read(f"raw_generations_{label}.json")
    ])
    scores = pd.DataFrame(read("judge_scores.json"))
    vram = pd.DataFrame(read("vram_measurements.json"))

    df = generations.merge(scores, on=["query_id", "model_label"], how="left")   # left, not inner: a missing score must surface below
    df = df.merge(vram, on="model_label", how="left")

    # Completeness is checked before the exclusion so that a stale score file is
    # caught even if the gap sits in an excluded query.
    unscored = df[df["judge_total"].isna()]
    if len(unscored) > 0:
        pairs = sorted(zip(unscored["query_id"], unscored["model_label"]))
        raise ValueError(f"Missing judge scores for {len(pairs)} responses: {pairs}")

    df = df[~df["query_id"].isin(excluded_queries)]
    return df.sort_values(["query_id", "model_label"]).reset_index(drop=True)


df = load_results(RESULTS_DIR, MODELS, EXCLUDED_QUERIES)

print(f"rows:     {len(df)}")
print(f"queries:  {df['query_id'].nunique()}   (excluded: {', '.join(EXCLUDED_QUERIES)})")
print(f"models:   {sorted(df['model_label'].unique())}")
print(f"columns:  {list(df.columns)}")
df.head()

rows:     351
queries:  117   (excluded: q068, q070, q113)
models:   ['base', 'student', 'teacher']
columns:  ['model_label', 'query_id', 'query_category', 'query_mood', 'sentence_count', 'query_text', 'response_text', 'generated_tokens', 'generation_time_sec', 'tokens_per_sec', 'word_count', 'acknowledgement', 'structured_steps', 'closing', 'tone', 'judge_total', 'vram_peak_gb']


,model_label,query_id,query_category,query_mood,sentence_count,query_text,response_text,generated_tokens,generation_time_sec,tokens_per_sec,word_count,acknowledgement,structured_steps,closing,tone,judge_total,vram_peak_gb
0,base,q001,Orders & Shipping,polite,2,"Guten Tag, ich wollte fragen ob Sie auch in di...",Vielen Dank fuer Ihre Anfrage. Hier ist eine k...,77,1.713,45.0,47,1,1,1,1,4,1.15
1,student,q001,Orders & Shipping,polite,2,"Guten Tag, ich wollte fragen ob Sie auch in di...",Vielen Dank fuer Ihre Anfrage.\n\n1. Die Liefe...,132,2.967,44.5,80,1,1,1,1,4,1.15
2,teacher,q001,Orders & Shipping,polite,2,"Guten Tag, ich wollte fragen ob Sie auch in di...",Guten Tag! Vielen Dank fuer Ihre Anfrage. Die ...,142,8.366,17.0,58,1,1,1,1,4,4.68
3,base,q002,Orders & Shipping,polite,1,"Koennten Sie mir sagen, bis zu welcher Uhrzeit...",Vielen Dank fuer Ihre Anfrage. Bitte bestellt ...,22,0.557,39.5,13,1,0,0,0,1,1.15
4,student,q002,Orders & Shipping,polite,1,"Koennten Sie mir sagen, bis zu welcher Uhrzeit...",Vielen Dank fuer Ihre Anfrage. Die Ware wird g...,57,1.287,44.3,33,1,0,1,1,3,1.15


## 2. Overview

Descriptive comparison of all three models, across the four format criteria and the practical cost metrics.<br>
No tests here. Absolute compliance rates are not testable, because the ideal rate for `structured_steps` is unknown.<br>
The tested comparison is agreement with the Teacher, in the sections below.

Computed over the same 117 queries as the agreement analysis.

In [3]:
def overview_table(df, models, criteria):
    """Aggregate descriptive metrics per model.

    Args:
        df: Long-format frame with one row per (query_id, model_label).
        models: Model labels, in display order.
        criteria: Binary criteria columns to average.

    Returns:
        DataFrame indexed by model_label with the mean format score, the four
        criteria as fractions, mean word count, mean throughput and peak VRAM.
    """
    rows = []
    for label in models:
        subset = df[df["model_label"] == label]
        rows.append({
            "model_label":    label,
            "format_score":   subset["judge_total"].mean(),
            **{c: subset[c].mean() for c in criteria},
            "word_count":     subset["word_count"].mean(),
            "tokens_per_sec": subset["tokens_per_sec"].mean(),
            "vram_peak_gb":   subset["vram_peak_gb"].iloc[0],   # one measurement per model, identical in every row
        })
    return pd.DataFrame(rows).set_index("model_label")


DISPLAY_NAMES = {
    "base":             "Base (1B)",
    "teacher":          "Teacher (7B+LoRA)",
    "student":          "Student (1B)",
    "acknowledgement":  "Acknowledgement",
    "structured_steps": "Structured Steps",
    "closing":          "Closing",
    "tone":             "Professional Tone",
}

overview = overview_table(df, MODELS, CRITERIA)

col = 20
print(f"{'Metric':<26}" + "".join(f"{DISPLAY_NAMES[m]:>{col}}" for m in MODELS))
print("-" * (26 + col * len(MODELS)))

print(f"{'Format Score (0-4)':<26}" + "".join(f"{overview.loc[m, 'format_score']:>{col}.2f}" for m in MODELS))
for criterion in CRITERIA:
    label = "  " + DISPLAY_NAMES[criterion]
    print(f"{label:<26}" + "".join(f"{overview.loc[m, criterion]:>{col}.1%}" for m in MODELS))

print(f"{'Avg Word Count':<26}" + "".join(f"{overview.loc[m, 'word_count']:>{col}.1f}" for m in MODELS))
print(f"{'Avg Tokens/sec':<26}" + "".join(f"{overview.loc[m, 'tokens_per_sec']:>{col}.1f}" for m in MODELS))
print(f"{'Peak VRAM (GB)':<26}" + "".join(f"{overview.loc[m, 'vram_peak_gb']:>{col}.2f}" for m in MODELS))

Metric                               Base (1B)   Teacher (7B+LoRA)        Student (1B)
--------------------------------------------------------------------------------------
Format Score (0-4)                        2.09                3.50                3.27
  Acknowledgement                        70.1%               75.2%               89.7%
  Structured Steps                       54.7%               85.5%               71.8%
  Closing                                31.6%               95.7%               94.0%
  Professional Tone                      53.0%               94.0%               71.8%
Avg Word Count                            79.0                63.7                58.0
Avg Tokens/sec                            45.3                15.8                45.3
Peak VRAM (GB)                            1.15                4.68                1.15


## 3. Agreement with the Teacher

For every query and every criterion, the Teacher's label is the reference.<br>
The Base and Student labels are compared against it, one query at a time.

Where a model deviates, the direction is recorded as well:

- `over`: the model applies the rule, the Teacher does not
- `under`: the Teacher applies the rule, the model does not

The direction cannot be derived from the aggregate rates in the overview.<br>
A gap of 73% against 88% may be 18 missing applications and none in excess, or 30 missing and 12 in excess.<br>
Only the query-level comparison separates the two.

In [4]:
def build_agreement(df, criteria, reference_model, compared_models):
    """Compare each model's binary labels against the reference model, query by query.

    Args:
        df: Long-format frame with one row per (query_id, model_label).
        criteria: Names of the binary criteria columns to compare.
        reference_model: Label of the model whose decisions define the target.
        compared_models: Labels of the models to be measured against the reference.

    Returns:
        DataFrame with one row per (query_id, criterion, model_label), carrying
        model_score, reference_score, match (0/1) and direction
        ("match", "over", "under").

    Raises:
        ValueError: If the reference model is absent from `df`.
    """
    if reference_model not in df["model_label"].values:
        raise ValueError(f"Reference model '{reference_model}' not present in the data.")

    wide = df.pivot(index="query_id", columns="model_label", values=criteria)

    records = []
    for criterion in criteria:
        reference = wide[(criterion, reference_model)]
        for model in compared_models:
            score = wide[(criterion, model)]
            match = (score == reference)
            records.append(pd.DataFrame({
                "query_id":        wide.index,
                "criterion":       criterion,
                "model_label":     model,
                "model_score":     score.values,
                "reference_score": reference.values,
                "match":           match.astype(int).values,
                # "over" and "under" are only defined where the labels differ.
                "direction": np.where(match, "match",
                                      np.where(score == 1, "over", "under")),
            }))

    return pd.concat(records, ignore_index=True)


agreement = build_agreement(df, CRITERIA, REFERENCE_MODEL, COMPARED_MODELS)

print(f"rows: {len(agreement)}   "
      f"({agreement['query_id'].nunique()} queries x {len(CRITERIA)} criteria x {len(COMPARED_MODELS)} models)")
agreement.head(8)

rows: 936   (117 queries x 4 criteria x 2 models)


,query_id,criterion,model_label,model_score,reference_score,match,direction
0,q001,acknowledgement,base,1,1,1,match
1,q002,acknowledgement,base,1,1,1,match
2,q003,acknowledgement,base,1,1,1,match
3,q004,acknowledgement,base,1,0,0,over
4,q005,acknowledgement,base,1,1,1,match
5,q006,acknowledgement,base,1,1,1,match
6,q007,acknowledgement,base,0,1,0,under
7,q008,acknowledgement,base,0,1,0,under


## 4. Agreement Rates

The share of queries on which a model makes the same call as the Teacher, per criterion, with a 95% interval.

The interval states how precisely the rate is known.<br>
Measured on a different set of queries the number would come out slightly differently.<br>
This answers Question 1: a high rate with a narrow interval means the behaviour transferred.<br>
A wide interval means the data cannot tell.

Wilson intervals are used rather than the textbook normal approximation.<br>
The latter produces bounds above 100% for rates close to the ceiling.

In [5]:
def agreement_rates(agreement, criteria, compared_models, confidence):
    """Aggregate per-query matches into an agreement rate per criterion and model.

    Args:
        agreement: Frame from `build_agreement`, one row per decision.
        criteria: Criteria names, in the order every table should display them.
        compared_models: Model labels to report, in display order.
        confidence: Confidence level for the interval, e.g. 0.95.

    Returns:
        DataFrame with one row per (criterion, model_label) and the columns
        matches, n, rate, ci_low, ci_high. Rates and bounds are fractions in [0, 1].
    """
    counts = (
        agreement
        .groupby(["criterion", "model_label"])["match"]
        .agg(matches="sum", n="count")
        .reset_index()
    )

    intervals = [
        stats.binomtest(int(row.matches), int(row.n))     # Wilson: criteria near the ceiling would otherwise exceed 100%
        .proportion_ci(confidence_level=confidence, method="wilson")
        for row in counts.itertuples()
    ]

    counts["rate"] = counts["matches"] / counts["n"]
    counts["ci_low"] = [i.low for i in intervals]
    counts["ci_high"] = [i.high for i in intervals]

    # groupby sorts alphabetically; restore the reading order used in every table.
    counts["criterion"] = pd.Categorical(counts["criterion"], categories=criteria, ordered=True)
    counts["model_label"] = pd.Categorical(counts["model_label"], categories=compared_models, ordered=True)

    return counts.sort_values(["criterion", "model_label"]).reset_index(drop=True)


rates = agreement_rates(agreement, CRITERIA, COMPARED_MODELS, CONFIDENCE)

print(f"{'Criterion':<20} {'Model':<9} {'Agreement':>10}  {'95% CI':^18} {'Matches':>10}")
print("-" * 72)
for row in rates.itertuples():
    print(f"{row.criterion:<20} {row.model_label:<9} {row.rate:>10.1%}  "
          f"[{row.ci_low:>6.1%}, {row.ci_high:>6.1%}] {row.matches:>7}/{row.n}")

Criterion            Model      Agreement        95% CI          Matches
------------------------------------------------------------------------
acknowledgement      base           59.0%  [ 49.9%,  67.5%]      69/117
acknowledgement      student        73.5%  [ 64.9%,  80.7%]      86/117
structured_steps     base           59.0%  [ 49.9%,  67.5%]      69/117
structured_steps     student        69.2%  [ 60.4%,  76.9%]      81/117
closing              base           30.8%  [ 23.1%,  39.6%]      36/117
closing              student        91.5%  [ 85.0%,  95.3%]     107/117
tone                 base           55.6%  [ 46.5%,  64.2%]      65/117
tone                 student        69.2%  [ 60.4%,  76.9%]      81/117


## 5. Difference in Agreement

How much closer to the Teacher the Student sits than the Base model, per criterion.<br>
Reported in percentage points with a 95% confidence interval.

This answers Question 2.<br>
An interval that excludes zero means the improvement is established, one that includes zero means it is not.<br>
The width separates the two reasons a result can fail: a small difference, or too few queries to resolve it.

The interval is obtained by bootstrapping over queries.<br>
Both models are resampled with the same draw, so each query keeps its pair of decisions together.

In [6]:
def _paired_matches(agreement, criterion, baseline, model):
    """Return the aligned 0/1 match vectors of two models for one criterion.

    Args:
        agreement: Frame from `build_agreement`, one row per decision.
        criterion: Criterion to extract.
        baseline: Model label whose vector is returned first.
        model: Model label whose vector is returned second.

    Returns:
        Two numpy arrays of equal length, both ordered by query_id, so that
        position i refers to the same query in both.
    """
    subset = agreement[agreement["criterion"] == criterion]
    wide = subset.pivot(index="query_id", columns="model_label", values="match")
    return wide[baseline].to_numpy(), wide[model].to_numpy()


def agreement_differences(agreement, criteria, baseline, model, n_bootstrap, confidence, seed):
    """Paired difference in agreement rate per criterion, with a bootstrap interval.

    Args:
        agreement: Frame from `build_agreement`.
        criteria: Criteria to report, in display order.
        baseline: Model label the difference is measured against.
        model: Model label whose improvement over the baseline is reported.
        n_bootstrap: Number of resampling draws.
        confidence: Confidence level for the interval, e.g. 0.95.
        seed: Seed for the resampling, so the bounds are reproducible.

    Returns:
        DataFrame with one row per criterion and the columns rate_baseline,
        rate_model, difference, ci_low, ci_high. All are fractions; the
        difference and its bounds may be negative.
    """
    rng = np.random.default_rng(seed)
    lower_pct = (1 - confidence) / 2 * 100
    upper_pct = (1 + confidence) / 2 * 100

    rows = []
    for criterion in criteria:
        base_match, model_match = _paired_matches(agreement, criterion, baseline, model)

        # One draw index for both models: each query keeps its pair of decisions together.
        draws = rng.integers(0, len(base_match), size=(n_bootstrap, len(base_match)))
        resampled = model_match[draws].mean(axis=1) - base_match[draws].mean(axis=1)
        ci_low, ci_high = np.percentile(resampled, [lower_pct, upper_pct])

        rows.append({
            "criterion":     criterion,
            "rate_baseline": base_match.mean(),
            "rate_model":    model_match.mean(),
            "difference":    model_match.mean() - base_match.mean(),
            "ci_low":        ci_low,
            "ci_high":       ci_high,
        })

    return pd.DataFrame(rows)


baseline, student = COMPARED_MODELS        # ["base", "student"], in that order

differences = agreement_differences(
    agreement, CRITERIA, baseline, student, N_BOOTSTRAP, CONFIDENCE, RANDOM_SEED
)

print(f"{'Criterion':<20} {'Base':>7} {'Student':>9} {'Difference':>12}  {'95% CI':^22}")
print("-" * 76)
for row in differences.itertuples():
    print(f"{row.criterion:<20} {row.rate_baseline:>7.1%} {row.rate_model:>9.1%} "
          f"{row.difference * 100:>+9.1f} pp  "
          f"[{row.ci_low * 100:>+6.1f}, {row.ci_high * 100:>+6.1f}] pp")

Criterion               Base   Student   Difference          95% CI        
----------------------------------------------------------------------------
acknowledgement        59.0%     73.5%     +14.5 pp  [  +6.0,  +23.1] pp
structured_steps       59.0%     69.2%     +10.3 pp  [  -1.7,  +22.2] pp
closing                30.8%     91.5%     +60.7 pp  [ +50.4,  +70.1] pp
tone                   55.6%     69.2%     +13.7 pp  [  +3.4,  +23.9] pp


## 6. Significance of the Difference

An exact McNemar test per criterion, on the same paired agreement data.

McNemar looks only at the queries where exactly one of the two models matched the Teacher.<br>
Queries on which both agreed, or both disagreed, say nothing about a difference and are ignored.<br>
The count of informative queries is reported alongside the p-value.<br>
Where it is small, no test can find anything, however many queries were evaluated in total.

Four criteria mean four tests, which raises the chance of one small p-value appearing by luck.<br>
The Holm correction adjusts for that. Both the raw and the adjusted value are shown.

In [7]:
def _holm_adjust(pvalues):
    """Adjust p-values for multiple testing using the Holm step-down procedure.

    Holm is preferred over Bonferroni here: it controls the same error rate but
    rejects more often, because only the smallest p-value carries the full penalty.

    Args:
        pvalues: 1-D array of raw p-values.

    Returns:
        Array of adjusted p-values, in the order of the input, each capped at 1.0.
    """
    m = len(pvalues)
    adjusted = np.empty(m)
    running_max = 0.0

    for rank, index in enumerate(np.argsort(pvalues)):
        running_max = max(running_max, (m - rank) * pvalues[index])   # adjusted values must not decrease along the sorted order
        adjusted[index] = min(running_max, 1.0)

    return adjusted


def mcnemar_tests(agreement, criteria, baseline, model):
    """Exact McNemar test per criterion on the paired agreement with the reference.

    Args:
        agreement: Frame from `build_agreement`.
        criteria: Criteria to test, in display order.
        baseline: Model label the comparison is made against.
        model: Model label whose improvement over the baseline is tested.

    Returns:
        DataFrame with one row per criterion and the columns favours_model,
        favours_baseline, n_discordant, p_value, p_holm.
    """
    rows = []
    for criterion in criteria:
        base_match, model_match = _paired_matches(agreement, criterion, baseline, model)

        favours_model = int(np.sum((base_match == 0) & (model_match == 1)))
        favours_baseline = int(np.sum((base_match == 1) & (model_match == 0)))
        n_discordant = favours_model + favours_baseline

        # Without discordant pairs the test has nothing to work on; 1.0 is the neutral result.
        p_value = (
            stats.binomtest(favours_model, n_discordant, 0.5).pvalue
            if n_discordant > 0 else 1.0
        )

        rows.append({
            "criterion":        criterion,
            "favours_model":    favours_model,
            "favours_baseline": favours_baseline,
            "n_discordant":     n_discordant,
            "p_value":          p_value,
        })

    result = pd.DataFrame(rows)
    result["p_holm"] = _holm_adjust(result["p_value"].to_numpy())
    return result


tests = mcnemar_tests(agreement, CRITERIA, baseline, student)

print(f"{'':<20}{'N only student':>16}{'N only base':>13}{'N Total':>13}{'':>9}{'':>10}")
print(f"{'Criterion':<20}{'agrees':>16}{'agrees':>13}{'informative':>13}{'p':>9}{'p (Holm)':>10}")
print("-" * 81)
for row in tests.itertuples():
    print(f"{row.criterion:<20}{row.favours_model:>16}{row.favours_baseline:>13}"
          f"{row.n_discordant:>13}{row.p_value:>9.4f}{row.p_holm:>10.4f}")

                      N only student  N only base      N Total                   
Criterion                     agrees       agrees  informative        p  p (Holm)
---------------------------------------------------------------------------------
acknowledgement                   24            7           31   0.0033    0.0100
structured_steps                  32           20           52   0.1263    0.1263
closing                           74            3           77   0.0000    0.0000
tone                              27           11           38   0.0139    0.0277


## 7. Agreement Results

The two questions side by side, per criterion.

**Question 1, similarity to the Teacher.**<br>
The Student's agreement rate with its interval, plus the direction of the remaining deviations.<br>
`N only student applied rule` counts queries where the Student meets the criterion and the Teacher does not.<br>
`N only Teacher applied rule` is the reverse case.

**Question 2, improvement over the Base model.**<br>
The paired difference with its interval, the number of queries the test could use, and the adjusted p-value.

In [8]:
def agreement_summary(rates, differences, tests, agreement, criteria, baseline, model):
    """Combine rates, differences, tests and deviation directions into one table.

    Args:
        rates: Output of `agreement_rates`.
        differences: Output of `agreement_differences`.
        tests: Output of `mcnemar_tests`.
        agreement: Frame from `build_agreement`, used for the direction counts.
        criteria: Criteria in display order.
        baseline: Model label used as the comparison point.
        model: Model label whose behaviour is being reported.

    Returns:
        DataFrame indexed by criterion, carrying the agreement rates and intervals
        of both models, the agreement expected from their marginal rates alone, the
        deviation counts of `model` against the reference, the paired difference with
        its interval, and the test results.
    """
    rates_wide = rates.pivot(index="criterion", columns="model_label",
                             values=["rate", "ci_low", "ci_high"])
    rates_wide.index = rates_wide.index.astype(str)   # drop the categorical dtype so the joins below align on plain labels

    deviations = pd.crosstab(
        agreement.loc[agreement["model_label"] == model, "criterion"],
        agreement.loc[agreement["model_label"] == model, "direction"],
    ).reindex(columns=["over", "under"], fill_value=0)

    # Agreement that follows from the marginal rates alone, if both models decided independently.
    marginals = agreement.groupby(["criterion", "model_label"])[["model_score", "reference_score"]].mean()

    table = pd.DataFrame(index=pd.Index(criteria, name="criterion"))

    for label, prefix in [(baseline, "base"), (model, "student")]:
        table[f"{prefix}_rate"] = rates_wide[("rate", label)]
        table[f"{prefix}_ci_low"] = rates_wide[("ci_low", label)]
        table[f"{prefix}_ci_high"] = rates_wide[("ci_high", label)]

        p_model = marginals.xs(label, level="model_label")["model_score"]
        p_reference = marginals.xs(label, level="model_label")["reference_score"]
        table[f"{prefix}_expected"] = p_model * p_reference + (1 - p_model) * (1 - p_reference)

    table["student_over"] = deviations["over"]
    table["student_under"] = deviations["under"]

    table = table.join(
        differences.set_index("criterion")[["difference", "ci_low", "ci_high"]]
        .rename(columns={"ci_low": "diff_ci_low", "ci_high": "diff_ci_high"})
    )
    return table.join(tests.set_index("criterion")[["n_discordant", "p_value", "p_holm"]])


summary = agreement_summary(rates, differences, tests, agreement, CRITERIA, baseline, student)

print("Question 1 — similarity to the Teacher\n")
print(f"{'':<20}{'':>15}{'expected by':>14}{'':>20}{'N only student':>16}{'N only Teacher':>16}")
print(f"{'Criterion':<20}{'Student agrees':>15}{'chance':>14}{'95% CI':>20}{'applied rule':>16}{'applied rule':>16}")
print("-" * 101)
for criterion, row in summary.iterrows():
    print(f"{criterion:<20}{row.student_rate:>15.1%}{row.student_expected:>14.1%}"
          f"{f'[{row.student_ci_low:.1%}, {row.student_ci_high:.1%}]':>20}"
          f"{int(row.student_over):>16}{int(row.student_under):>16}")

print("\n\nQuestion 2 — improvement over the Base model\n")
print(f"{'Criterion':<20}{'Base':>9}{'Student':>10}{'Difference':>14}{'95% CI':>22}"
      f"{'informative':>13}{'p (Holm)':>11}")
print("-" * 99)
for criterion, row in summary.iterrows():
    print(f"{criterion:<20}{row.base_rate:>9.1%}{row.student_rate:>10.1%}"
          f"{row.difference * 100:>+11.1f} pp"
          f"{f'[{row.diff_ci_low * 100:+.1f}, {row.diff_ci_high * 100:+.1f}] pp':>22}"
          f"{int(row.n_discordant):>13}{row.p_holm:>11.4f}")

Question 1 — similarity to the Teacher

                                      expected by                      N only student  N only Teacher
Criterion            Student agrees        chance              95% CI    applied rule    applied rule
-----------------------------------------------------------------------------------------------------
acknowledgement               73.5%         70.0%      [64.9%, 80.7%]              24               7
structured_steps              69.2%         65.5%      [60.4%, 76.9%]              10              26
closing                       91.5%         90.3%      [85.0%, 95.3%]               4               6
tone                          69.2%         69.2%      [60.4%, 76.9%]               5              31


Question 2 — improvement over the Base model

Criterion                Base   Student    Difference                95% CI  informative   p (Holm)
--------------------------------------------------------------------------------------------------

## 8. Response Length

Distribution of response lengths in words, recorded during inference in Notebook 04.

Median and range are shown alongside the mean.<br>
A single runaway response would lift the mean without moving the median.

In [9]:
def distribution_table(df, column, models):
    """Summarise a per-response metric per model.

    Args:
        df: Long-format frame with one row per (query_id, model_label).
        column: Name of the numeric column to summarise.
        models: Model labels, in display order.

    Returns:
        DataFrame indexed by model_label with mean, median, min and max.
    """
    return (
        df.groupby("model_label")[column]
        .agg(["mean", "median", "min", "max"])
        .reindex(models)                       # groupby sorts alphabetically
    )


words = distribution_table(df, "word_count", MODELS)

print(f"{'Model':<12}{'Mean':>9}{'Median':>9}{'Min':>7}{'Max':>7}")
print("-" * 44)
for label, row in words.iterrows():
    print(f"{label:<12}{row['mean']:>9.1f}{row['median']:>9.0f}{row['min']:>7.0f}{row['max']:>7.0f}")

Model            Mean   Median    Min    Max
--------------------------------------------
base             79.0       67      6    223
teacher          63.7       63     10    144
student          58.0       52     14    177


## 9. Throughput

Inference speed in tokens per second.

Wall-clock time per query is not used, because it is confounded by output length.<br>
A model that generates more tokens looks slower even at identical speed.<br>
Tokens per second measures throughput independently of response length.

In [10]:
speed = distribution_table(df, "tokens_per_sec", MODELS)

print(f"{'Model':<12}{'Mean':>9}{'Median':>9}{'Min':>8}{'Max':>8}")
print("-" * 46)
for label, row in speed.iterrows():
    print(f"{label:<12}{row['mean']:>9.1f}{row['median']:>9.1f}{row['min']:>8.1f}{row['max']:>8.1f}")

Model            Mean   Median     Min     Max
----------------------------------------------
base             45.3     45.9    36.2    46.8
teacher          15.8     15.8    13.3    17.0
student          45.3     45.4    43.0    46.7


## 10. Peak VRAM

Peak GPU memory during inference, recorded in Notebook 04.

All three models were loaded with `load_in_4bit=True`.<br>
The comparison therefore reflects model size and not the quantisation format.

In [11]:
vram = df.groupby("model_label")["vram_peak_gb"].first().reindex(MODELS)

bar_width = 28
for label, gb in vram.items():
    filled = round((gb / vram.max()) * bar_width)
    print(f"{label:<10} {'█' * filled}{'░' * (bar_width - filled)}  {gb:.2f} GB")

base       ███████░░░░░░░░░░░░░░░░░░░░░  1.15 GB
teacher    ████████████████████████████  4.68 GB
student    ███████░░░░░░░░░░░░░░░░░░░░░  1.15 GB


## 11. Why Not BERTScore

BERTScore measures semantic similarity between a generated response and a reference text.<br>
It answers the question: does this response say the same thing as the reference?

That is not what this project trains for.<br>
The Teacher was fine-tuned for **style and format transfer**: structured steps, polite tone, acknowledgement, a closing offer.<br>
These are structural and tonal properties, not semantic content.

A base model that writes a correct but unstructured prose response would score high on BERTScore.<br>
It would score low on every criterion this project actually cares about.<br>
Including the metric would obscure the evaluation rather than clarify it.

The agreement analysis measures the property the training targeted.<br>
Word count, throughput and VRAM measure the practical trade-offs of distillation.

## 12. Save Results

Two files.<br>
`eval_summary.json` keeps the per-model descriptive metrics in the structure the README already refers to.<br>
`agreement_summary.json` holds the agreement analysis together with the settings it was produced under.

The second file makes the numbers citable later without re-running the notebook.

In [12]:
summary_records = [
    {
        "model_label":          label,
        "format_score_mean":    round(overview.loc[label, "format_score"], 2),
        "acknowledgement_pct":  round(overview.loc[label, "acknowledgement"] * 100, 1),
        "structured_steps_pct": round(overview.loc[label, "structured_steps"] * 100, 1),
        "closing_pct":          round(overview.loc[label, "closing"] * 100, 1),
        "tone_pct":             round(overview.loc[label, "tone"] * 100, 1),
        "word_count_mean":      round(overview.loc[label, "word_count"], 1),
        "tokens_per_sec_mean":  round(overview.loc[label, "tokens_per_sec"], 1),
        "vram_peak_gb":         float(overview.loc[label, "vram_peak_gb"]),
    }
    for label in MODELS
]

agreement_output = {
    "reference_model":  REFERENCE_MODEL,
    "compared_models":  COMPARED_MODELS,
    "n_queries":        int(df["query_id"].nunique()),
    "excluded_queries": EXCLUDED_QUERIES,
    "confidence":       CONFIDENCE,
    "n_bootstrap":      N_BOOTSTRAP,
    "random_seed":      RANDOM_SEED,
    "criteria":         summary.reset_index().round(4).to_dict(orient="records"),
}

for filename, payload in [("eval_summary.json", summary_records),
                          ("agreement_summary.json", agreement_output)]:
    with open(os.path.join(RESULTS_DIR, filename), "w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)
    print(f"Saved: {filename}")

Saved: eval_summary.json
Saved: agreement_summary.json
